# Bet sizing and risk of ruin

A trading bot with a real edge can still go broke. Not because the edge is wrong, but because the bankroll is finite: a run of losses can reach zero before the edge has time to pay off. This post asks three questions. Given an edge, how many bets does it take to realize it? Can clever bet sizing beat a house edge? And how large should each bet be? A Monte Carlo random walk makes the answers concrete.

## Gambler's ruin

Take even-money bets: win one unit with probability $p$, lose one with probability $q = 1 - p$. Start with a bankroll of $u$ units, bet one unit at a time, and play forever. The probability of eventually going broke is

$$P(\text{ruin}) = \begin{cases} \left(\dfrac{q}{p}\right)^{u} & p > q \\[2mm] 1 & p \le q. \end{cases}$$

Two things fall out at once. With no edge or a negative one ($p \le q$), ruin is **certain** — the only question is when, and smaller bets merely slow the walk down. With a positive edge ($p > q$), ruin decays geometrically in the bankroll $u$: doubling your bankroll roughly *squares* your odds of survival.

If instead you stop at a target of $T$ units, the ruin probability starting from $u$ is

$$P(\text{ruin}) = \frac{(q/p)^{T} - (q/p)^{u}}{(q/p)^{T} - 1} \quad (p \ne q), \qquad P(\text{ruin}) = \frac{T - u}{T} \quad (p = q).$$

## How big should a bet be?

Fixed-unit betting ignores that a growing bankroll can afford larger bets. The **Kelly criterion** sizes each bet as a fraction of the *current* bankroll. For even money the optimal fraction is

$$f^{*} = p - q,$$

and for general net odds $b$ it is $f^{*} = \dfrac{bp - q}{b}$. Kelly maximizes the long-run growth rate

$$g(f) = p\ln(1 + f) + q\ln(1 - f),$$

which peaks at $f^{*}$ and turns **negative** once $f$ exceeds about $2f^{*}$: bet too big and a positive edge still compounds you toward zero. Betting a fixed fraction of Kelly — a half, say — gives up a little growth for a large cut in variance and drawdown.

The lesson underneath all of it: sizing cannot manufacture an edge. With $p \le q$ every strategy loses in the end. Sizing only decides how you spend an edge you already have.

In [1]:
import numpy as np


def simulate(p, u, n_bets, n_paths, fraction=None, target=None, seed=0):
    rng = np.random.default_rng(seed)
    wins = rng.random((n_paths, n_bets)) < p
    ruin_level = 0.0 if fraction is None else 1.0
    paths = np.empty((n_paths, n_bets + 1))
    paths[:, 0] = float(u)
    bankroll = np.full(n_paths, float(u))
    done = np.zeros(n_paths, dtype=bool)
    for t in range(n_bets):
        if fraction is None:
            step = np.where(wins[:, t], 1.0, -1.0)
        else:
            step = fraction * bankroll * np.where(wins[:, t], 1.0, -1.0)
        bankroll = np.where(done, bankroll, bankroll + step)
        done = done | (bankroll <= ruin_level)
        if target is not None:
            done = done | (bankroll >= target)
        paths[:, t + 1] = bankroll
    ruin = float(np.mean(np.min(paths, axis=1) <= ruin_level))
    return paths, ruin

## Watching the walk

Below are sample bankroll paths under flat one-unit betting, starting from $u = 25$ units over $300$ bets. The dashed red line at zero is ruin — once a path touches it, that path is out for good. Drag the slider to change the edge. At a house edge ($p < 0.5$) almost every path finds zero; nudge $p$ above $0.5$ and most paths pull away, yet some still bust early. That early-bust risk, not the long-run average, is the whole problem.

In [2]:
import plotly.graph_objects as go

edges = [0.47, 0.49, 0.51, 0.53]
u0 = 25
n_bets = 300
n_show = 30

fig = go.Figure()
counts = []
for pe in edges:
    paths, ruin = simulate(pe, u0, n_bets, 400, seed=1)
    counts.append(ruin)
    x = list(range(n_bets + 1))
    for i in range(n_show):
        fig.add_trace(go.Scatter(x=x, y=paths[i].tolist(), mode="lines",
                                 line=dict(width=1), opacity=0.7,
                                 visible=(pe == 0.51), showlegend=False,
                                 hoverinfo="skip"))

fig.add_hline(y=0, line=dict(color="crimson", dash="dash"))
steps = []
for idx, pe in enumerate(edges):
    vis = [idx * n_show <= j < (idx + 1) * n_show for j in range(len(edges) * n_show)]
    steps.append(dict(method="update",
                      args=[{"visible": vis},
                            {"title": f"Flat betting, p = {pe} (ruin within {n_bets} bets: {counts[idx]:.0%})"}],
                      label=str(pe)))
fig.update_layout(
    sliders=[dict(active=2, currentvalue={"prefix": "win probability p = "}, steps=steps)],
    title=f"Flat betting, p = 0.51 (ruin within {n_bets} bets: {counts[2]:.0%})",
    xaxis_title="bets",
    yaxis_title="bankroll (units)",
    margin=dict(t=60),
)
fig.show()

## Bet size and the Kelly optimum

With a real edge the question becomes *how much*. The left panel fixes the edge and sweeps the flat bet size: betting $b$ units at a time turns a $u$-unit bankroll into only $u/b$ bets' worth of cushion, so ruin climbs as $(q/p)^{u/b}$ — the markers are simulated, the dashed line is that formula. Bigger bets mean more ruin, even with the edge fully intact.

The right panel switches to betting a fixed *fraction* of the bankroll and plots the long-run growth rate $g(f) = p\ln(1+f) + q\ln(1-f)$. It peaks at the Kelly fraction $f^{*} = p - q$ (the red star) and crosses zero near $2f^{*}$: past that point, more aggression buys negative growth. Slide the edge to watch the optimum move right and the safe range widen.

In [3]:
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

edges2 = [0.51, 0.52, 0.54, 0.56]
u2 = 40
bet_sizes = [1, 2, 4, 5, 8, 10]
fracs = np.linspace(0.001, 0.30, 120)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Flat betting: ruin vs bet size",
                                    "Fractional betting: growth vs fraction"))
per = 4
for pe in edges2:
    q = 1 - pe
    ruin_sim = [simulate(pe, u2 // b, 4000, 2000, seed=2)[1] for b in bet_sizes]
    ruin_th = [(q / pe) ** (u2 / b) for b in bet_sizes]
    show = (pe == 0.52)
    fig.add_trace(go.Scatter(x=bet_sizes, y=ruin_sim, mode="markers",
                             marker=dict(size=9, color="#1f77b4"), visible=show,
                             showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=bet_sizes, y=ruin_th, mode="lines",
                             line=dict(color="#999999", dash="dash"), visible=show,
                             showlegend=False), row=1, col=1)
    g = pe * np.log(1 + fracs) + q * np.log(1 - fracs)
    fstar = pe - q
    fig.add_trace(go.Scatter(x=fracs.tolist(), y=g.tolist(), mode="lines",
                             line=dict(color="#1f77b4"), visible=show,
                             showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=[fstar], y=[float(pe * np.log(1 + fstar) + q * np.log(1 - fstar))],
                             mode="markers", marker=dict(size=13, color="crimson", symbol="star"),
                             visible=show, showlegend=False), row=1, col=2)

fig.add_hline(y=0, line=dict(color="#bbbbbb", width=1), row=1, col=2)
steps = []
for idx, pe in enumerate(edges2):
    vis = [idx == j // per for j in range(len(edges2) * per)]
    steps.append(dict(method="update",
                      args=[{"visible": vis},
                            {"title": f"Edge p = {pe}:  Kelly f* = {pe - (1 - pe):.2f}"}],
                      label=str(pe)))
fig.update_layout(
    sliders=[dict(active=1, currentvalue={"prefix": "win probability p = "}, steps=steps)],
    title=f"Edge p = 0.52:  Kelly f* = {0.52 - 0.48:.2f}",
    margin=dict(t=90),
)
fig.update_xaxes(title_text="bet size (units)", row=1, col=1)
fig.update_yaxes(title_text="ruin probability", row=1, col=1)
fig.update_xaxes(title_text="fraction of bankroll f", row=1, col=2)
fig.update_yaxes(title_text="growth rate g(f)", row=1, col=2)
fig.show()

## What sizing can and cannot do

Put the pieces together. An edge is necessary: with $p \le q$ ruin is certain, and no bet size changes the destination — only the scenery on the way down. Given an edge, the bankroll is the real asset, since ruin falls geometrically in it, and bet size is the dial that trades growth against survival. Flat bets that are too large hand the edge back to variance; fractional Kelly extracts the most growth, but only up to $f^{*} = p - q$, beyond which more aggression compounds toward zero. Most practitioners bet a fraction of Kelly, because the growth curve is flat near its peak while the drawdowns are not. For a trading bot the takeaway is blunt: prove the edge first, size to survive the losing streaks a finite bankroll guarantees, and never mistake a bigger bet for a better one.

**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-bet-sizing-and-risk-of-ruin.ipynb)**